# FreshMart Lab 2: Data Wrangler + Feature Engineering
**Microsoft Fabric Data Science**

ใช้ Data Wrangler แล้ว **Add code to notebook** ได้จริง

| ส่วน | ทำอะไร |
| --- | --- |
| **A** | Wrangler กับธุรกรรม (format / filter / group by) → ใส่โค้ดกลับ notebook |
| **B** | รันสัญญาฟีเจอร์สมาชิก → ได้ Silver + `feature_params.json` สำหรับ Lab 3–4 |

Lab 4 ต้องใช้ params จากชุดฝึก — ส่วน B จึงล็อกสเกลให้ทั้งคลาส


In [ ]:
import json
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    try:
        frame = spark.read.table(table_name).toPandas()
        print(f"Loaded Spark table {table_name}: {len(frame):,} rows")
        return frame
    except Exception as exc:
        print(f"Spark table '{table_name}' unavailable ({exc}). Falling back to CSV.")
        return load_csv(file_name)


### ส่วน A — โหลดธุรกรรมสำหรับ Data Wrangler

สุ่ม 500 แถวให้ UI ตอบสนองเร็วขึ้น


In [ ]:
df = load_table_or_csv("bronze_transactions", "freshmart_transactions.csv")
df = df.sample(n=500, random_state=1).reset_index(drop=True)
print("Sample for Data Wrangler:", df.shape)
df.head(4)


### ส่วน A — เปิด Data Wrangler แล้วใส่โค้ดกลับ notebook

1. รอ kernel ว่าง แล้วแท็บ **Home** → **Data Wrangler** → เลือก `df`
2. ดู Summary ของ `WasteCost`
3. **Format** คอลัมน์ `Category` (Capitalize all words) → Apply
4. **Filter** `StoreType` = `Express` และ/หรือ **Sort** `WasteCost` Descending → Apply
5. ลบขั้น Filter/Sort จาก Cleaning steps ถ้าต้องการใช้ทั้งตัวอย่าง แล้ว **Group by** `Category` + aggregate `WasteCost` (Mean)
6. กด **Add code to notebook** — วางโค้ดในเซลล์ว่างด้านล่าง แล้วรัน

ตัวอย่างโครงฟังก์ชันหลังได้โค้ดจาก Wrangler:


In [ ]:
def summarize_waste(frame: pd.DataFrame) -> pd.DataFrame:
    """ตัวอย่างหลัง Export จาก Data Wrangler — ปรับให้ตรงโค้ดที่คุณได้จาก UI"""
    out = frame.groupby(["Category"]).agg(WasteCost_mean=("WasteCost", "mean")).reset_index()
    return out

print(summarize_waste(df))


### ส่วน B — โหลดสมาชิกสำหรับโมเดล

(ทางเลือก) เปิด Wrangler กับ `df_cust` เพื่อลอง Fill Age / One-hot / Scale  
สำหรับส่งงาน Lab 3–4 ให้รันเซลล์สัญญาฟีเจอร์ถัดไป เพื่อได้ `feature_params.json` ชุดเดียวกันทั้งคลาส


In [ ]:
df_cust = load_table_or_csv("bronze_customers", "freshmart_customers.csv")
print("Customers DataFrame loaded:", df_cust.shape)
print("Age missing:", int(df_cust["Age"].isna().sum()))
df_cust.head(3)


### ส่วน B — รันสัญญาฟีเจอร์ที่เตรียมให้


In [ ]:
from dataclasses import asdict, dataclass
from typing import Any
import logging

logger = logging.getLogger(__name__)

ID_COLUMN = 'CustomerID'
TARGET_COLUMN = 'Churn'
CATEGORICAL_COLUMNS = ('MembershipTier', 'Gender')
IMPUTE_MEDIAN_COLUMNS = ('Age',)
SCALE_COLUMNS = ('MonetaryTotal', 'AvgBasketSize', 'RecencyDays', 'TenureMonths')
DUMMY_COLUMNS = ('MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
PASSTHROUGH_COLUMNS = ('Frequency', 'ComplaintCount')
FEATURE_COLUMNS = ('Age', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount', 'MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
REQUIRED_RAW_COLUMNS = ('CustomerID', 'Age', 'Gender', 'MembershipTier', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount')

@dataclass(frozen=True)
class FeatureParams:
    """Fitted preprocessing parameters reused at scoring time.

    Attributes:
        age_median: Median Age computed from the training customers.
        scale_mins: Per-column minimum used for min-max scaling.
        scale_maxs: Per-column maximum used for min-max scaling.
        dummy_columns: One-hot columns the model expects, in order.
        feature_columns: Final model input columns, in order.
    """

    age_median: float
    scale_mins: dict[str, float]
    scale_maxs: dict[str, float]
    dummy_columns: list[str]
    feature_columns: list[str]

    def to_dict(self) -> dict[str, Any]:
        """Serialize parameters to a JSON-friendly dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> FeatureParams:
        """Create parameters from a dictionary.

        Args:
            payload: Mapping produced by ``to_dict``.

        Returns:
            Validated ``FeatureParams``.

        Raises:
            ValueError: If required keys are missing.
        """
        required = {
            "age_median",
            "scale_mins",
            "scale_maxs",
            "dummy_columns",
            "feature_columns",
        }
        missing = required - set(payload)
        if missing:
            raise ValueError(f"FeatureParams missing keys: {sorted(missing)}")
        return cls(
            age_median=float(payload["age_median"]),
            scale_mins={k: float(v) for k, v in payload["scale_mins"].items()},
            scale_maxs={k: float(v) for k, v in payload["scale_maxs"].items()},
            dummy_columns=list(payload["dummy_columns"]),
            feature_columns=list(payload["feature_columns"]),
        )


def validate_raw_customers(df: pd.DataFrame, *, require_target: bool = True) -> pd.DataFrame:
    """Validate and normalize a raw FreshMart customer frame.

    Args:
        df: Raw customer records from CSV or ``bronze_customers``.
        require_target: When True, require the ``Churn`` column.

    Returns:
        Copy with numeric columns coerced.

    Raises:
        ValueError: If required columns are missing.
    """
    required = REQUIRED_RAW_COLUMNS + ((TARGET_COLUMN,) if require_target else ())
    _require_columns(df, required, frame_name="customer frame")
    numeric_cols = (
        "Age",
        "TenureMonths",
        "RecencyDays",
        "Frequency",
        "MonetaryTotal",
        "AvgBasketSize",
        "ComplaintCount",
    )
    if require_target:
        numeric_cols = numeric_cols + (TARGET_COLUMN,)
    return _numeric_copy(df, numeric_cols)


def _require_columns(df: pd.DataFrame, columns: tuple[str, ...], *, frame_name: str) -> None:
    """Raise if expected columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {missing}")


def _numeric_copy(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Return a copy with selected columns coerced to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def fit_preprocessor(df_raw: pd.DataFrame) -> FeatureParams:
    """Fit imputation and scaling parameters on training customers.

    Args:
        df_raw: Raw customer DataFrame including ``Churn``.

    Returns:
        Fitted parameters that must be reused for scoring.
    """
    df = validate_raw_customers(df_raw, require_target=True)
    age_median = float(df["Age"].median())
    if pd.isna(age_median):
        raise ValueError("Cannot fit preprocessor: Age median is NaN")

    scale_mins: dict[str, float] = {}
    scale_maxs: dict[str, float] = {}
    for col in SCALE_COLUMNS:
        col_min = float(df[col].min())
        col_max = float(df[col].max())
        if col_max <= col_min:
            raise ValueError(f"Cannot scale {col}: min={col_min}, max={col_max}")
        scale_mins[col] = col_min
        scale_maxs[col] = col_max

    params = FeatureParams(
        age_median=age_median,
        scale_mins=scale_mins,
        scale_maxs=scale_maxs,
        dummy_columns=list(DUMMY_COLUMNS),
        feature_columns=list(FEATURE_COLUMNS),
    )
    logger.info(
        "Fitted feature params: age_median=%.1f, scale_columns=%s",
        params.age_median,
        list(SCALE_COLUMNS),
    )
    return params


def _one_hot_categories(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode membership and gender, keeping the expected columns."""
    encoded = pd.get_dummies(df, columns=list(CATEGORICAL_COLUMNS), drop_first=False)
    for col in DUMMY_COLUMNS:
        if col not in encoded.columns:
            encoded[col] = 0
    return encoded


def _min_max_scale(series: pd.Series, col_min: float, col_max: float) -> pd.Series:
    """Scale a series to [0, 1] using fitted min/max."""
    return (series - col_min) / (col_max - col_min + 1e-6)


def transform_customers(
    df_raw: pd.DataFrame,
    params: FeatureParams,
    *,
    require_target: bool = False,
) -> pd.DataFrame:
    """Apply the fitted FreshMart preprocessing contract.

    Args:
        df_raw: Raw customer records (training or scoring batch).
        params: Parameters from ``fit_preprocessor``.
        require_target: When True, keep and validate ``Churn``.

    Returns:
        Feature frame with ``CustomerID``, model columns, and optional ``Churn``.
    """
    df = validate_raw_customers(df_raw, require_target=require_target)
    work = df.copy()
    work["Age"] = work["Age"].fillna(params.age_median)

    work = _one_hot_categories(work)
    for col in SCALE_COLUMNS:
        work[col] = _min_max_scale(
            work[col],
            params.scale_mins[col],
            params.scale_maxs[col],
        )

    bool_cols = work.select_dtypes(include="bool").columns
    work[bool_cols] = work[bool_cols].astype(int)

    ordered = [ID_COLUMN, *params.feature_columns]
    if require_target or TARGET_COLUMN in work.columns:
        ordered.append(TARGET_COLUMN)
    missing = [col for col in ordered if col not in work.columns]
    if missing:
        raise ValueError(f"Transformed frame missing columns: {missing}")

    result = work[ordered].copy()
    feature_frame = result[list(params.feature_columns)]
    if feature_frame.isna().any().any():
        bad = feature_frame.columns[feature_frame.isna().any()].tolist()
        raise ValueError(f"NaN remaining in feature columns: {bad}")
    return result


def model_matrix(df_features: pd.DataFrame, params: FeatureParams) -> pd.DataFrame:
    """Return the model input matrix in signature order.

    Args:
        df_features: Output of ``transform_customers``.
        params: Fitted parameters.

    Returns:
        DataFrame with only model feature columns.
    """
    _require_columns(df_features, tuple(params.feature_columns), frame_name="feature frame")
    return df_features[list(params.feature_columns)].astype(float)


def save_feature_params(params: FeatureParams, path: str | Path) -> Path:
    """Write feature parameters to a JSON file.

    Args:
        params: Fitted parameters.
        path: Destination JSON path.

    Returns:
        Resolved output path.
    """
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(params.to_dict(), indent=2), encoding="utf-8")
    logger.info("Saved feature params to %s", out)
    return out


def load_feature_params(path: str | Path) -> FeatureParams:
    """Load feature parameters from a JSON file.

    Args:
        path: JSON path written by ``save_feature_params``.

    Returns:
        Fitted parameters.

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"Feature params not found: {src}")
    payload = json.loads(src.read_text(encoding="utf-8"))
    return FeatureParams.from_dict(payload)


In [ ]:
params = fit_preprocessor(df_cust)
df_clean = transform_customers(df_cust, params, require_target=True)
print(f"Cleaned DataFrame Shape: {df_clean.shape}")
print("Feature columns:", params.feature_columns)
print(f"Age median used for imputation: {params.age_median:.1f}")
display(df_clean.head(5))


### ส่วน B — บันทึก Silver table และ params สำหรับ Lab 4


In [ ]:
params_json = json.dumps(params.to_dict(), indent=2)
local_dir = Path("labs/data/.local")
local_dir.mkdir(parents=True, exist_ok=True)
(local_dir / "feature_params.json").write_text(params_json, encoding="utf-8")

saved_to_files = False
for writer in ("notebookutils", "mssparkutils"):
    try:
        fs = __import__(writer).fs
        fs.mkdirs("Files/params")
        fs.put("Files/params/feature_params.json", params_json, True)
        saved_to_files = True
        print(f"Saved Files/params/feature_params.json via {writer}")
        break
    except Exception as exc:
        print(f"{writer} unavailable: {exc}")

if not saved_to_files:
    print("Saved local feature_params.json only")

try:
    spark.createDataFrame(df_clean).write.format("delta").mode("overwrite").saveAsTable("silver_customer_features")
    print("Saved silver_customer_features to Lakehouse")
except Exception as exc:
    out = local_dir / "silver_customer_features.csv"
    df_clean.to_csv(out, index=False)
    print(f"Spark write unavailable ({exc}). Wrote {out}")


In [ ]:
if df_clean["Age"].isna().sum() != 0:
    raise AssertionError("หลังเตรียมฟีเจอร์แล้ว Age ไม่ควรว่าง")
if not set(["MembershipTier_Bronze", "Gender_F"]).issubset(df_clean.columns):
    raise AssertionError("ขาดคอลัมน์ one-hot ที่โมเดลคาดหวัง — รันเซลล์ fit/transform อีกครั้ง")
if df_clean["MonetaryTotal"].max() > 1.000001:
    raise AssertionError("MonetaryTotal ควรอยู่ในช่วงประมาณ 0–1 หลัง scale จากชุดฝึก")
print("Lab 2 verification passed")
